# Face Identification Using PCA and k-Nearest Neighbours

This notebook develops a closed-set face-identification system for five known identities. Face images are represented with PCA-based Eigenfaces and classified using k-Nearest Neighbours (k-NN).

The workflow is developed incrementally so that data preparation, model selection, final testing, and error analysis remain clearly separated.

## 1. Dataset exploration

The project uses the **Labeled Faces in the Wild (LFW)** dataset provided through Scikit-learn. LFW contains cropped face photographs collected under unconstrained conditions, including changes in pose, expression, illumination, and image quality.

This section loads the complete dataset and inspects its class distribution before any identities are selected.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_lfw_people

RANDOM_STATE = 42
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

np.random.seed(RANDOM_STATE)
plt.style.use("seaborn-v0_8-whitegrid")

print(f"Random seed: {RANDOM_STATE}")
print(f"Results directory: {RESULTS_DIR.resolve()}")

### Load LFW

`fetch_lfw_people` downloads the dataset during the first run and then reuses the local cached copy. Images are loaded in grayscale at half resolution to keep the experiment computationally manageable. No minimum image count is applied at this stage because the original identity distribution must be inspected first.

In [ ]:
lfw = fetch_lfw_people(
    data_home=DATA_DIR,
    funneled=True,
    resize=0.5,
    min_faces_per_person=0,
    color=False,
    download_if_missing=True,
)

images = lfw.images
features = lfw.data
labels = lfw.target
identity_names = lfw.target_names

print("LFW loaded successfully")

### Dataset dimensions

The following checks confirm the number of images, identities, image dimensions, and flattened features available for later PCA processing.

In [ ]:
n_images, image_height, image_width = images.shape
n_identities = len(identity_names)

dataset_summary = pd.Series(
    {
        "Number of images": n_images,
        "Number of identities": n_identities,
        "Image height": image_height,
        "Image width": image_width,
        "Features per image": features.shape[1],
        "Minimum pixel value": float(images.min()),
        "Maximum pixel value": float(images.max()),
    },
    name="Value",
).to_frame()

display(dataset_summary)

### Identity distribution

LFW is strongly imbalanced: many people have only one image, while a small number have many photographs. The counts below help define a reproducible rule for selecting five identities with enough samples for training and testing.

In [ ]:
image_counts = np.bincount(labels, minlength=n_identities)
identity_counts = (
    pd.DataFrame(
        {
            "identity_id": np.arange(n_identities),
            "identity": identity_names,
            "image_count": image_counts,
        }
    )
    .sort_values("image_count", ascending=False)
    .reset_index(drop=True)
)

display(identity_counts.head(15))
print(f"Identities with at least 50 images: {(image_counts >= 50).sum()}")
print(f"Identities with only one image: {(image_counts == 1).sum()}")

In [ ]:
top_identities = identity_counts.head(15).sort_values("image_count")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_identities["identity"], top_identities["image_count"], color="#31688e")
ax.set_title("Fifteen Most Represented Identities in LFW")
ax.set_xlabel("Number of images")
ax.set_ylabel("Identity")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "lfw_identity_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

### Representative face images

One deterministic sample is displayed for each of the ten most represented identities. These examples provide an initial view of the variation in pose, expression, background, and illumination.

In [ ]:
top_ten = identity_counts.head(10)
sample_indices = [np.flatnonzero(labels == identity_id)[0] for identity_id in top_ten["identity_id"]]

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for ax, sample_index in zip(axes.ravel(), sample_indices):
    ax.imshow(images[sample_index], cmap="gray")
    ax.set_title(identity_names[labels[sample_index]], fontsize=9)
    ax.axis("off")

fig.suptitle("Representative Images from the Most Frequent LFW Identities", fontsize=14)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "lfw_sample_faces.png", dpi=200, bbox_inches="tight")
plt.show()

### Exploration checkpoint

The next stage will use these counts to select five sufficiently represented identities and construct a balanced subset. No training, test splitting, PCA fitting, or classification is performed in this section.